# Tutorial: Pandas Exercise Solutions

Audience:
- Instructors or learners reviewing worked solutions for the short exercises in [`EXERCISES.md`](EXERCISES.md).

Prerequisites:
- Basic Python syntax.
- A working pandas environment with Excel support.

Learning goals:
- Load tabular data from CSV and Excel.
- Filter, summarize, reshape, and clean data with pandas.
- Compare common approaches such as boolean masks vs `.query()` and `apply()` vs vectorization.


## Outline

1. Setup and shared data
2. Basic pandas objects and indexing
3. Cleaning and missing data
4. Aggregation, pivoting, and reshaping
5. `apply`, vectorization, and pipelines
6. CSV import and a mini analysis


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
metadata = pd.read_excel(DATA_DIR / "patient_metadata.xlsx")
experiment = pd.read_excel(DATA_DIR / "patient_experiment.xlsx")
missing = pd.read_csv(DATA_DIR / "missing_values.csv")
patients = experiment.merge(metadata, on="patient", how="left")

print("metadata", metadata.shape)
print("experiment", experiment.shape)
print("missing", missing.shape)
patients.head()


## 1. Create a Series

Simple labeled `Series` construction and access.


In [ ]:
s = pd.Series([3, 5, 8, 13, 21], index=["a", "b", "c", "d", "e"])
print(s)
print("mean", s.mean())
print("min", s.min())
print("max", s.max())
print("label b", s["b"])
print("position 2", s.iloc[2])


## 2. Build a DataFrame

Build a small `DataFrame` and derive one new column.


In [ ]:
df = pd.DataFrame(
    {
        "name": ["Alice", "Bob", "Carol"],
        "age": [28, 34, 41],
        "city": ["Brussels", "Antwerp", "Ghent"],
    }
)
df["age_in_10_years"] = df["age"] + 10
df


## 3. Inspect Patient Metadata

Read the spreadsheet and inspect its structure.


In [ ]:
print(metadata.shape)
print(metadata.dtypes)
metadata.head()


## 4. Filter Rows with Boolean Masks

The worksheet mentions age; in this repository the closest numeric example is `temperature`. Use the merged patient table.


In [ ]:
high_temp = patients[patients["temperature"] > 38.5]
condition_a = patients[patients["condition"] == "A"]
high_temp_f = patients[(patients["temperature"] > 38.5) & (patients["gender"] == "F")]

print("high temperature rows:", len(high_temp))
print("condition A rows:", len(condition_a))
high_temp_f.head()


## 5. Compare `.loc` and `.iloc`

Retrieve similar subsets by label and by integer position.


In [ ]:
loc_subset = metadata.loc[0:3, ["patient", "gender"]]
iloc_subset = metadata.iloc[0:4, [0, 1]]

print("loc subset")
display(loc_subset)
print("iloc subset")
display(iloc_subset)


## 6. Missing-Value Count

Count null values per column.


In [ ]:
missing.isna().sum().sort_values(ascending=False)


## 7. Fill or Drop Missing Values

Show two cleanup strategies on the same dataset.


In [ ]:
missing_drop = missing.dropna()
missing_fill = missing.fillna(
    {
        "int_data": missing["int_data"].median(),
        "float_data": missing["float_data"].mean(),
        "category_data": "Unknown",
        "string_data": "missing",
    }
)

print("original", missing.shape)
print("dropna", missing_drop.shape)
print("fillna", missing_fill.shape)
missing_fill


## 8. Clean Column Names

Normalize deliberately messy headers to lowercase `snake_case`.


In [ ]:
messy = metadata.rename(
    columns={"patient": "Patient ID", "gender": "Gender Code", "condition": "Condition Group"}
)
messy.columns = (
    messy.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
messy.head()


## 9. Query Syntax Practice

Reproduce a boolean-mask filter with `.query()`.


In [ ]:
mask_result = patients[(patients["temperature"] > 38.5) & (patients["gender"] == "F")]
query_result = patients.query("temperature > 38.5 and gender == 'F'")
print(len(mask_result), len(query_result))
query_result.head()


## 10. Sort and Rank

Sort experiment rows by temperature and rank them.


In [ ]:
ranked = patients.sort_values("temperature", ascending=False).copy()
ranked["temp_rank"] = ranked["temperature"].rank(method="dense", ascending=False)
ranked[["patient", "temperature", "temp_rank"]].head(10)


## 11. Group and Aggregate

Group the merged data by condition and summarize temperature.


In [ ]:
patients.groupby("condition")["temperature"].agg(["count", "mean", "std"]) 


## 12. Build a Pivot Table

Summarize mean temperature by condition and gender.


In [ ]:
pd.pivot_table(
    patients,
    index="condition",
    columns="gender",
    values="temperature",
    aggfunc="mean",
)


## 13. `pivot()` vs `pivot_table()`

Use a small synthetic example to show why duplicate keys break `pivot()`.


In [ ]:
duplicates = pd.DataFrame(
    {
        "patient": [1, 1, 1, 2],
        "measure": ["temp", "temp", "dose", "temp"],
        "value": [38.1, 38.5, 2.0, 37.9],
    }
)

try:
    duplicates.pivot(index="patient", columns="measure", values="value")
except ValueError as exc:
    print("pivot error:", exc)

pd.pivot_table(duplicates, index="patient", columns="measure", values="value", aggfunc="mean")


## 14. Long to Wide

Reshape a tidy long table to wide format.


In [ ]:
long_df = pd.DataFrame(
    {
        "patient": [1, 1, 2, 2],
        "day": ["day1", "day2", "day1", "day2"],
        "temperature": [38.3, 38.5, 37.9, 38.0],
    }
)
long_df.pivot(index="patient", columns="day", values="temperature")


## 15. Wide to Long

Melt repeated measurement columns into tidy form.


In [ ]:
wide_df = pd.DataFrame(
    {
        "patient": [1, 2, 3],
        "day1": [38.3, 37.9, 38.1],
        "day2": [38.5, 38.0, 38.2],
        "day3": [38.1, 37.8, 38.0],
    }
)
wide_df.melt(id_vars=["patient"], var_name="day", value_name="temperature")


## 16. Classify with `apply()`

Map temperatures to categories using a custom function.


In [ ]:
def classify_temperature(value):
    if value < 38.0:
        return "normal"
    if value < 38.5:
        return "elevated"
    return "high"

classified = patients.copy()
classified["temp_band"] = classified["temperature"].apply(classify_temperature)
classified[["patient", "temperature", "temp_band"]].head(10)


## 17. Replace `apply()` with Vectorization

Solve the same classification problem with `np.select()`.


In [ ]:
vectorized = patients.copy()
vectorized["temp_band"] = np.select(
    [
        vectorized["temperature"] < 38.0,
        vectorized["temperature"] < 38.5,
    ],
    ["normal", "elevated"],
    default="high",
)
vectorized[["patient", "temperature", "temp_band"]].head(10)


## 18. Build a `.pipe()` Workflow

Chain together a small cleanup-and-summary workflow.


In [ ]:
def clean_columns(df):
    result = df.copy()
    result.columns = result.columns.str.strip().str.lower()
    return result


def keep_high_temps(df, threshold=38.0):
    return df[df["temperature"] > threshold]

summary = (
    patients.pipe(clean_columns)
    .pipe(keep_high_temps, threshold=38.0)
    .groupby("condition")["temperature"]
    .mean()
    .sort_values(ascending=False)
)
summary


## 19. Read Non-Standard CSV Files

The repository includes a generator script. This solution shows the import side on a tiny semicolon-separated example.


In [ ]:
from io import StringIO

csv_text = """patient;temperature;condition
1;38.3;A
2;37.9;B
"""
semicolon_df = pd.read_csv(StringIO(csv_text), sep=";")
semicolon_df


## 20. Mini Analysis

Small end-to-end example: merge, summarize, and plot mean temperature by condition and gender.


In [ ]:
analysis = patients.copy()
summary = (
    analysis.groupby(["condition", "gender"])["temperature"]
    .mean()
    .unstack()
    .round(2)
)
ax = summary.plot(kind="bar", title="Mean temperature by condition and gender", ylabel="temperature")
summary


## Notes for Instructors

- Exercises 4, 9, 10, 11, 12, 16, 17, 18, and 20 are adapted to the actual repository columns: `patient`, `gender`, `condition`, `dose`, `date`, and `temperature`.
- Exercises 13, 14, and 15 use small synthetic tables because they isolate reshaping concepts more clearly than the raw workbook data.
- The cells are intentionally short so they can be demonstrated live and modified during class.
